### Playing around with the requests library

In [5]:
import requests

routeURL = "https://pt.mytransitride.com/api/Route"
vehicleStatusesURL = "https://pt.mytransitride.com/api/VehicleStatuses"

route = requests.get("https://pt.mytransitride.com/api/Route")

route.json()

[{'routeID': 2623,
  'routeNumber': 1425,
  'routeCode': '4',
  'routeName': 'Weller',
  'color': '0070C0',
  'headSign': '4 - PTBO Museum',
  'patternID': 11158,
  'patternCode': '4-East-42 (4)',
  'patternName': '4-East-42 (4)',
  'patternNumber': 10780,
  'directionID': 18,
  'directionName': 'East',
  'serviceID': None,
  'serviceName': None,
  'arrivalTimes': None},
 {'routeID': 2624,
  'routeNumber': 1426,
  'routeCode': '5',
  'routeName': 'The Parkway',
  'color': 'DED30B',
  'headSign': '5 - PTBO Terminal',
  'patternID': 11171,
  'patternCode': '5-North-36 (5)',
  'patternName': '5-North-36 (5)',
  'patternNumber': 8311,
  'directionID': 1,
  'directionName': 'North',
  'serviceID': None,
  'serviceName': None,
  'arrivalTimes': None},
 {'routeID': 2625,
  'routeNumber': 1427,
  'routeCode': '6',
  'routeName': 'Sherbrooke',
  'color': '066A40',
  'headSign': '6 - Trent U Bata',
  'patternID': 11190,
  'patternCode': '6-East-92 (6)',
  'patternName': '6-East-92 (6)',
  'patte

In [23]:
with open("patterns.txt") as patterns:
    patternIds = patterns.read()

parameters = {
    "patternIds": patternIds
}

current_logs = requests.get(vehicleStatusesURL, params = parameters)

current_logs_json = requests.get(vehicleStatusesURL, params = parameters).json()

current_logs_json

[{'patternId': 11215,
  'name': '50',
  'iconUrl': 'https://pt.mapstrat.com/images/vehicle.png',
  'vehicleCapacityIndicator': '',
  'isLight': False,
  'isMedium': False,
  'isFull': False,
  'headsignText': '8 - Fleming College',
  'vehicleId': 3,
  'lastUpdate': '2026-10-07T23:53:42',
  'lat': 44.2747205,
  'lng': -78.3245363,
  'velocity': 47,
  'bearing': 161,
  'vehicleStateId': 1,
  'bypassDailyTripId': None},
 {'patternId': 11226,
  'name': '20',
  'iconUrl': 'https://pt.mapstrat.com/images/vehicle.png',
  'vehicleCapacityIndicator': '',
  'isLight': False,
  'isMedium': False,
  'isFull': False,
  'headsignText': '9 - Trent U Gzowski',
  'vehicleId': 6,
  'lastUpdate': '2026-10-07T23:53:30',
  'lat': 44.313031,
  'lng': -78.331027,
  'velocity': 37,
  'bearing': 70,
  'vehicleStateId': 1,
  'bypassDailyTripId': None},
 {'patternId': 11256,
  'name': '22',
  'iconUrl': 'https://pt.mapstrat.com/images/vehicle.png',
  'vehicleCapacityIndicator': '',
  'isLight': False,
  'isMediu

---
Now that I know what the data looks like, it is time to create a system for fetching and organizing it.

I first want to manipulate the returned string and create a dictionary containing:

```
{
    "pattern_id",
    "vehicle_id",
    "route_id",
    "bearing",
    "headsign_text",
    "velocity",
    "lat",
    "lng",
    "last_update"
}
```

This will serve my facts table. My dimension table will be served by the reponse from api/Route, and change infrequently.

In [16]:
current_logs_json[0]

{'patternId': 11215,
 'name': '50',
 'iconUrl': 'https://pt.mapstrat.com/images/vehicle.png',
 'vehicleCapacityIndicator': '',
 'isLight': False,
 'isMedium': False,
 'isFull': False,
 'headsignText': '8 - Fleming College',
 'vehicleId': 3,
 'lastUpdate': '2026-10-07T23:11:05',
 'lat': 44.3576763,
 'lng': -78.2907495,
 'velocity': 0,
 'bearing': 1,
 'vehicleStateId': 1,
 'bypassDailyTripId': None}

In [32]:
from datetime import datetime, timezone
from zoneinfo import ZoneInfo

PTBO_tz = ZoneInfo("America/Toronto")



# print(len(current_logs_json))

def get_current_logs():

    vehicle_logs = []

    current_logs_json = requests.get(vehicleStatusesURL, params = parameters).json()

    for i in range(len(current_logs_json)):
        last_update = datetime.fromisoformat(current_logs_json[i]["lastUpdate"]).replace(tzinfo=timezone.utc)
        last_update = last_update.astimezone(PTBO_tz)
        vehicle_logs.append(
            {
                "pattern_id": current_logs_json[i]["patternId"],
                "vehicle_id": current_logs_json[i]["vehicleId"],
                "headsign_text": current_logs_json[i]["headsignText"],
                "velocity": current_logs_json[i]["velocity"],
                "bearing": current_logs_json[i]["bearing"],
                "lat": current_logs_json[i]["lat"],
                "lng": current_logs_json[i]["lng"],
                "last_update": last_update.isoformat()
            }
        )

    return vehicle_logs

print(get_current_logs()[0])

{'pattern_id': 11226, 'vehicle_id': 6, 'headsign_text': '9 - Trent U Gzowski', 'velocity': 42, 'bearing': 43, 'lat': 44.3478972, 'lng': -78.3012173, 'last_update': '2026-10-07T20:08:53-04:00'}


---
Next, I'm going to write the logs to a file.

In [39]:
with open(f"data/logs_write{datetime.now().strftime("%Y-%m-%dT%H-%M-%S")}", "a+") as f:
    for item in get_current_logs():
        f.writelines(f"{item}\n")